## Demand Trends & Seasonal Patterns

Demand forecasting sits at the core of supply chain planning — inventory positioning, carrier capacity commitments, and warehouse staffing all depend on understanding how order volume moves over time. This notebook examines demand patterns at the department and category level, with a focus on identifying whether volume trends are consistent or whether certain product lines have seasonal spikes that require differentiated planning cycles.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data
from src.viz_utils import plot_monthly_order_volume

pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = load_data('../data/dataco_supply_chain.csv')

In [ ]:
fig = plot_monthly_order_volume(df)
fig.show()

The 90-day moving average shows the macro trend while the 30-day highlights shorter-cycle fluctuations. Persistent divergence between the two signals an inflection — either a structural demand shift or a one-off event that hasn't mean-reverted.

In [ ]:
dept_monthly = (
    df.assign(month=df['order_date'].dt.to_period('M'))
    .groupby(['department_name', 'month'])['order_quantity']
    .sum()
    .unstack('department_name')
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, 6))
dept_monthly.plot(ax=ax, linewidth=1.5)
ax.set_title('Monthly Order Quantity by Department')
ax.set_xlabel('')
ax.set_ylabel('Units Ordered')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
cat_monthly = (
    df.assign(month=df['order_date'].dt.month)
    .groupby(['category_name', 'month'])['order_quantity']
    .sum()
    .unstack('month')
    .fillna(0)
)

# Limit to top 15 categories by total volume
top_cats = cat_monthly.sum(axis=1).nlargest(15).index
cat_monthly = cat_monthly.loc[top_cats]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    cat_monthly,
    ax=ax,
    cmap='YlOrRd',
    fmt='.0f',
    annot=True,
    linewidths=0.3,
    cbar_kws={'label': 'Units Ordered'}
)
ax.set_title('Order Volume by Category × Month (Top 15 Categories)')
ax.set_xlabel('Month')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

Several categories show clear month-over-month concentration — this is not random noise, it reflects real seasonality or promotional calendar patterns. Categories with high Q4 concentration (months 10–12) are candidates for earlier inventory pre-positioning and carrier capacity locking.

In [ ]:
df['year'] = df['order_date'].dt.year

yoy = (
    df.groupby(['department_name', 'year'])['order_quantity']
    .sum()
    .unstack('year')
)

years = sorted(yoy.columns)
if len(years) >= 2:
    yoy['yoy_growth_pct'] = ((yoy[years[-1]] - yoy[years[-2]]) / yoy[years[-2]] * 100).round(1)
    print(yoy.sort_values('yoy_growth_pct', ascending=False).to_string())

In [ ]:
if 'yoy_growth_pct' in yoy.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#4A6FA5' if v >= 0 else '#E8735A' for v in yoy['yoy_growth_pct']]
    yoy['yoy_growth_pct'].sort_values().plot.barh(ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('YoY Order Growth (%)')
    ax.set_title('Year-over-Year Order Growth by Department')
    plt.tight_layout()
    plt.show()

> **Insight:** Departments with negative YoY growth aren't necessarily declining — check whether the dataset has full coverage for both years before drawing conclusions. Partial-year data in the most recent year will mechanically depress YoY numbers. Where growth is genuinely flat or negative, the question for planning is whether to rebalance inventory allocation or reopen carrier rate negotiations to reduce fixed cost exposure in those lines.